# Module 315 — Practical Introduction to MCP

This notebook turns HTB Academy's practical MCP example into a small, annotated local study exercise. It demonstrates server registration, client discovery and calls, and the JSON-RPC messages those calls represent.

**Safety choices:** the server binds only to 127.0.0.1; file reads and writes are constrained to a dedicated temporary folder; names are checked before becoming paths; and the example tool limits input size. The course sample is intentionally minimal and does not include those protections. This remains teaching code, not a production service.

The later MCP security sections build on this vocabulary. This notebook stays on the practical-introduction material.

## 1. Map the code to MCP concepts

- **Host:** the application coordinating the LLM interaction.
- **Client:** this notebook's Client, which speaks MCP to the server.
- **Server:** the FastMCP instance that implements the functions below.
- **Prompt:** a reusable text template selected by the user.
- **Resource:** information that the application can read, such as a file count or file contents.
- **Tool:** an operation that can change state; store_file writes a file.

A Python decorator such as @mcp.tool() registers the function below it as an MCP capability. FastMCP uses the function name, parameters, type hints, and docstring to describe that capability to clients.

In [ ]:
# Uncomment in the AI environment only if FastMCP is not already installed:
# %pip install fastmcp

from pathlib import Path
import re
import tempfile

from fastmcp import Client, FastMCP

# Use a fresh, dedicated directory so this exercise cannot write into arbitrary folders.
STUDY_DIR = Path(tempfile.mkdtemp(prefix="mcp-study-")).resolve()
HOST = "127.0.0.1"  # Loopback: local machine only.
PORT = 8000
MAX_FILE_BYTES = 10_000

mcp = FastMCP("MCP Study Server")

print(f"Study files will be kept under: {STUDY_DIR}")

## 2. Define a prompt

A prompt is a reusable template. The function does not execute the spelling check itself; it returns text that can be given to an LLM.

Read text: str aloud as “text, type string.” It is a Python type hint describing the expected input. It helps FastMCP describe the capability, but it does not replace validation.

In [ ]:
@mcp.prompt()
def spell_check(text: str) -> str:
    """Generate a user message asking for a spell check of input text."""
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    if len(text) > MAX_FILE_BYTES:
        raise ValueError("text is too long for this study example")

    # Return prompt content; the LLM performs the language task later.
    return (
        "Please check the following text for typos and grammatical errors:\n\n"
        f"{text}"
    )

## 3. Define read-only resources

The fixed resource URI resource://filecount returns how many study files exist. The URI template getfile://{file_name} contains a variable: requesting getfile://hello supplies hello as file_name.

The path check matters. The HTB demonstration interpolates a caller-provided filename into a path. In real code, a name such as ../../somewhere must not escape the intended directory. This study version accepts only a simple filename and resolves the final path under STUDY_DIR.

In [ ]:
def safe_study_path(file_name: str) -> Path:
    """Return a path inside STUDY_DIR only for a simple, safe base name."""
    if not re.fullmatch(r"[A-Za-z0-9_-]{1,64}", file_name):
        raise ValueError("file_name must contain only letters, digits, '_' or '-'")
    candidate = (STUDY_DIR / f"{file_name}.mcpfile").resolve()

    # Verify containment after path resolution, not just by inspecting the input.
    if candidate.parent != STUDY_DIR:
        raise ValueError("resolved path is outside the study directory")
    return candidate


@mcp.resource("resource://filecount")
def count_files() -> int:
    """Return the number of study files currently stored."""
    return sum(1 for _ in STUDY_DIR.glob("*.mcpfile"))


@mcp.resource("getfile://{file_name}")
def get_file(file_name: str) -> str:
    """Read one named file from the dedicated study directory."""
    path = safe_study_path(file_name)
    return path.read_text(encoding="utf-8")

## 4. Define a state-changing tool

A tool differs from the resources above because it writes data. It checks the filename and content size before writing. A real service also needs authorization, retention, and a decision about whether a person must approve the action.

In [ ]:
@mcp.tool()
def store_file(file_content: str, file_name: str) -> str:
    """Store a small text file in the dedicated MCP study directory."""
    if not isinstance(file_content, str):
        raise TypeError("file_content must be a string")
    if len(file_content.encode("utf-8")) > MAX_FILE_BYTES:
        raise ValueError("file content exceeds the study limit")

    path = safe_study_path(file_name)
    path.write_text(file_content, encoding="utf-8")
    return f"Stored {file_name}.mcpfile ({len(file_content)} characters)"

## 5. Use an in-memory server/client connection

The course starts an HTTP server so Wireshark can capture its traffic. In this notebook, we use FastMCP's in-memory transport instead: the client talks directly to this server object in the same process. This keeps the notebook interactive and avoids a blocking server cell or a second Python process. The message-shape examples below still illustrate JSON-RPC; they are not a packet capture.


## 6. Connect a client and discover the prompt

The FastMCP client uses asynchronous I/O. Read async def as asynchronous function definition. Await means pause this function until the operation finishes. The async with block opens and closes the client session cleanly. Client(mcp) selects FastMCP's in-memory connection to the server object defined above.


In [ ]:
client = Client(mcp)

async def inspect_prompt():
    async with client:
        prompts = await client.list_prompts()
        prompt_result = await client.get_prompt(
            'spell_check',
            {'text': 'Hello World!'},
        )
        prompt_text = prompt_result.messages[0].content.text

    print('Available prompts:', prompts)
    print('Prompt content:', prompt_text)

await inspect_prompt()

## 7. Call the tool, then read resources

The first client call asks the server to run store_file. That is a write action. The next calls read the count and file contents through resources. Keeping them separate highlights the distinction between a tool and a resource.

In [ ]:
async def store_and_read():
    async with client:
        tools = await client.list_tools()
        tool_result = await client.call_tool(
            "store_file",
            {"file_content": "Hello World!", "file_name": "hello"},
        )

        count_result = await client.read_resource("resource://filecount")
        file_result = await client.read_resource("getfile://hello")

    print("Available tools:", tools)
    print("Tool result:", tool_result)
    print("File count:", count_result[0].text)
    print("Read-back content:", file_result[0].text)

await store_and_read()

## 8. See the JSON-RPC shape

A request ID lets the client match a response to the request. Method names the operation, and params carries arguments when needed. A response repeats the same ID and provides either result or error. An initialization-complete notification has no ID because it expects no reply.

These dictionaries illustrate message structure; they do not send network traffic.

In [ ]:
import json

example_request = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "prompts/get",
    "params": {
        "name": "spell_check",
        "arguments": {"text": "Hello World!"},
    },
}

example_response = {
    "jsonrpc": "2.0",
    "id": 2,
    "result": {
        "messages": [
            {"role": "user", "content": {
                "type": "text", "text": "Please check Hello World!"
            }}
        ]
    },
}

initialized_notification = {
    "jsonrpc": "2.0",
    "method": "notifications/initialized",
}

print("Request:\n", json.dumps(example_request, indent=2))
print("Response:\n", json.dumps(example_response, indent=2))
print("Notification:\n", json.dumps(initialized_notification, indent=2))

## 9. Error and security observations

FastMCP can turn a raised exception into a JSON-RPC error associated with the request ID. Raw exception text can reveal paths or implementation details. Log diagnostics securely and return a restrained message to callers.

Review input validation, resolved path containment, authorization, tool side effects, error disclosure, secret handling, and network scope. A resource is read-only by protocol convention in this lesson, but the server must still enforce safe reads.

## COAE recall

- @mcp.prompt(), @mcp.resource(...), and @mcp.tool() register different MCP capability types.
- A URI template such as getfile://{file_name} has a parameter supplied by the client.
- list_* discovers capabilities; get_prompt, read_resource, and call_tool use them.
- async and await support non-blocking network operations.
- JSON-RPC request and response IDs correlate messages; they are not permission checks.
- A write-capable tool needs tighter validation and authorization than a read-only resource.
- HTB's minimal file example teaches protocol behavior. Do not copy unvalidated path construction into production.